# Native Rust + Candle FP32 T5 — 5-Text Colab CPU Benchmark

Runs `JayShah07/falconai-text-bullet-t5` natively in Rust/Candle with no Python/PyTorch in the inference process. Measures checkpoint size, RSS/peak RSS, encoder prefill, first decoder step, TTFT, mean/median/p95/max ITL, total latency and throughput, with live token streaming.

Candle's native T5 implementation supports safetensors and cached decoder generation. This first compatibility benchmark uses greedy decoding; it intentionally does not reproduce Hugging Face `no_repeat_ngram_size=3`, so exact text may differ.

In [ ]:
# CELL 0 — install Rust/build prerequisites
!apt-get update -qq
!apt-get install -y -qq build-essential pkg-config libssl-dev git curl
!command -v rustc >/dev/null 2>&1 || curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
import os
os.environ['PATH']=os.path.expanduser('~/.cargo/bin')+':'+os.environ['PATH']
!rustc --version
!cargo --version

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package pkg-config:amd64.
(Reading database ... 122797 files and directories currently installed.)
Preparing to unpack .../pkg-config_1.8.1-2build1_amd64.deb ...
Unpacking pkg-config:amd64 (1.8.1-2build1) ...
Setting up pkg-config:amd64 (1.8.1-2build1) ...
info: downloading installer
info: profile set to default
info: default host tuple is x86_64-unknown-linux-gnu
info: syncing channel updates for stable-x86_64-unknown-linux-gnu
info: latest update on 2026-09-03 for version 1.98.1 (48a229cea 2026-09-01)
info: downloading 6 components
      rustfmt installed                        2.37 MiB                         info: default toolchain set to stable-x86_64-unknown-linux-gnu

  stable-x86_64-unknown-linux-gnu installed - rustc 1.98.1 (48a229cea 2026-09-01)


Rust is ins

In [ ]:
# CELL 1 — create project
!rm -rf /content/candle_t5_native
!mkdir -p /content/candle_t5_native/src
%cd /content/candle_t5_native

/content/candle_t5_native


In [ ]:
# CELL 2 — pin Candle main revision and write Cargo.toml
import subprocess
REV=subprocess.check_output(['git','ls-remote','https://github.com/huggingface/candle.git','refs/heads/main'],text=True).split()[0]
print('Candle revision:',REV)
cargo=f"""[package]
name = \"candle-t5-native\"
version = \"0.1.0\"
edition = \"2021\"
[dependencies]
anyhow=\"1\"
serde={{version=\"1\",features=[\"derive\"]}}
serde_json=\"1\"
tokenizers=\"0.22\"
hf-hub=\"0.4\"
candle-core={{git=\"https://github.com/huggingface/candle.git\",rev=\"{REV}\"}}
candle-nn={{git=\"https://github.com/huggingface/candle.git\",rev=\"{REV}\"}}
candle-transformers={{git=\"https://github.com/huggingface/candle.git\",rev=\"{REV}\"}}
"""
open('/content/candle_t5_native/Cargo.toml','w').write(cargo)
print(cargo)

Candle revision: ddf1b879dc3a1760cbcb3f3c4a7c6467850cec4a
[package]
name = "candle-t5-native"
version = "0.1.0"
edition = "2021"
[dependencies]
anyhow="1"
serde={version="1",features=["derive"]}
serde_json="1"
tokenizers="0.22"
hf-hub="0.4"
candle-core={git="https://github.com/huggingface/candle.git",rev="ddf1b879dc3a1760cbcb3f3c4a7c6467850cec4a"}
candle-nn={git="https://github.com/huggingface/candle.git",rev="ddf1b879dc3a1760cbcb3f3c4a7c6467850cec4a"}
candle-transformers={git="https://github.com/huggingface/candle.git",rev="ddf1b879dc3a1760cbcb3f3c4a7c6467850cec4a"}



In [ ]:
# ============================================================
# CELL 3 — WRITE CORRECTED NATIVE RUST/CANDLE BENCHMARK
#
# FIXES:
# 1. Clear KV cache before every independent request
# 2. Safe HF-style no_repeat_ngram_size=3
# 3. Greedy decoding
# 4. Cached T5 decoder
# 5. Streaming
# 6. EOS tracking
# 7. TTFT / prefill / ITL / latency / memory metrics
# ============================================================

rust_source = r'''
use anyhow::{Context, Error as E, Result};

use candle_core::{
    DType,
    Device,
    Tensor,
};

use candle_nn::VarBuilder;

use candle_transformers::models::t5;

use hf_hub::{
    api::sync::Api,
    Repo,
    RepoType,
};

use serde::Serialize;

use std::{
    collections::HashSet,
    fs,
    io::{self, Write},
    path::Path,
    time::{Duration, Instant},
};

use tokenizers::Tokenizer;


// ============================================================
// CONFIG
// ============================================================

const MODEL_ID: &str =
    "JayShah07/falconai-text-bullet-t5";

const MAX_NEW_TOKENS: usize = 256;

const NO_REPEAT_NGRAM_SIZE: usize = 3;

const BULLET_TOKEN: &str =
    "<BULLET>";


// ============================================================
// EXACT TASK PROMPT
// ============================================================

const TASK_INSTRUCTION: &str = r#"Convert the following English text into concise bullet points containing all materially important information.

Follow these rules:

- Extract all important and independently useful points.
- The number of bullets must depend entirely on the information in the text.
- Never use a fixed number of bullets.
- Use one bullet for each distinct important point.
- Combine details that naturally belong together.
- Remove repetition, filler, metadata, boilerplate, and trivial details.
- Do not repeat the same information in multiple bullets.
- Preserve important names, dates, numbers, quantities, comparisons, causes, conditions, decisions, and conclusions.
- Do not add, infer, or assume information that is not supported by the source text.
- Do not turn contextual information into new advice or recommendations.
- Keep every bullet concise while preserving the original meaning.
- Return only bullet points.
- Start every bullet with "- ".

Text:"#;


fn build_text(text: &str) -> String {
    format!(
        "{TASK_INSTRUCTION}\n{}",
        text.trim()
    )
}


// ============================================================
// FIVE TEST TEXTS
// ============================================================

fn test_texts()
    -> Vec<(&'static str, &'static str)>
{
    vec![
        (
            "01_very_short",

            r#"Apex Systems reported quarterly revenue of $1.8 billion, up 9% year over year. Operating profit increased 6% to $240 million, while management maintained its full-year guidance."#,
        ),

        (
            "02_short",

            r#"Vertex Systems reported quarterly revenue of $3.8 billion, up 16% from the same period last year. Cloud-service revenue increased 28%, while legacy hardware sales declined 7%. Operating profit rose from $410 million to $475 million, although operating margin decreased from 18.1% to 17.4% because of higher infrastructure and energy costs. The company added 820,000 paying customers and said demand remained strong in North America and Asia."#,
        ),

        (
            "03_medium",

            r#"Northstar Technologies reported second-quarter revenue of $8.7 billion, an increase of 18% year over year. Growth was driven primarily by cloud infrastructure and enterprise subscriptions. Cloud revenue rose 31%, while the older hardware business declined 9%. Operating profit increased from $940 million to $1.12 billion, but operating margin declined from 21.4% to 20.1% because of higher data-center, energy and hiring costs. The company added 1.6 million subscription customers, taking the total to 12.4 million. Management raised full-year revenue guidance from $33 billion to between $34.5 billion and $35 billion and warned that foreign-exchange movements could reduce fourth-quarter revenue by approximately $300 million."#,
        ),

        (
            "04_long",

            r#"Meridian Group said annual revenue increased 14% to $21.6 billion as strong growth in digital services offset weaker traditional consulting demand. Digital-services revenue rose 27% and now represents 46% of group sales. Operating profit increased 11% to $2.4 billion, although operating margin slipped from 16.2% to 15.8% after increased spending on data centers, cybersecurity and artificial intelligence. Meridian signed 34 contracts worth more than $100 million, added approximately 18,000 employees primarily in India, Poland and Mexico, and reduced headcount in several higher-cost European offices. Free cash flow rose to $1.9 billion from $1.5 billion. The board approved a 12% dividend increase and a new $2 billion share-repurchase program. Management expects revenue growth of 9% to 11% next year but warned that customers in Germany and France are delaying discretionary technology projects. The company plans two new cloud facilities in Asia and also announced a machine-learning security platform for enterprise customers."#,
        ),

        (
            "05_very_long",

            r#"Orion Global reported full-year revenue of $47.3 billion, up 19%, after growth across cloud infrastructure, payments and enterprise software. Cloud infrastructure revenue increased 34% to $16.8 billion, supported by financial-services companies and artificial intelligence developers. Payments revenue rose 22% as transaction volume increased 18%, although consumer spending weakened in parts of Western Europe. Enterprise-software revenue increased 11%, while legacy on-premise licensing declined 13%. Operating profit increased from $5.1 billion to $6.0 billion, but operating margin fell from 22.8% to 22.1% because Orion spent heavily on data centers, AI accelerators and security infrastructure. Capital expenditure reached $7.4 billion versus $4.9 billion a year earlier. The company added 4.2 million subscription customers, bringing the total to 31.7 million, while enterprise retention remained above 95%. Orion signed 61 contracts worth more than $50 million, including nine worth more than $250 million each. The company also announced a restructuring of its consumer-device division, eliminating approximately 3,500 positions and consolidating five manufacturing sites into three. The program is expected to cost $420 million to $480 million but generate approximately $650 million in annual savings. Free cash flow increased 16% to $5.8 billion, the quarterly dividend was raised 10%, and another $4 billion was approved for share repurchases. Net debt declined to $8.1 billion from $9.6 billion. Orion expects next-year revenue growth of 12% to 14% and operating margin between 22% and 23%. Risks include weaker European enterprise spending, constrained accelerator supply, higher electricity costs and regulatory reviews of its payments business. Its order backlog is nevertheless 24% higher than a year ago. Finally, Orion introduced an enterprise security product combining automated threat detection, identity monitoring and incident-response tools, with premium customers receiving it from October 15 and general availability scheduled for November 3."#,
        ),
    ]
}


// ============================================================
// RESULT
// ============================================================

#[derive(Debug, Serialize, Clone)]
struct BenchResult {

    test: String,

    input_words: usize,
    input_tokens: usize,
    output_tokens: usize,

    reached_eos: bool,

    checkpoint_size_mb: f64,

    rss_mb: f64,
    peak_rss_mb: f64,

    encoder_prefill_seconds: f64,
    first_decoder_step_seconds: f64,

    ttft_seconds: f64,

    mean_itl_ms: f64,
    median_itl_ms: f64,
    p95_itl_ms: f64,
    max_itl_ms: f64,

    overall_latency_seconds: f64,

    decode_tokens_per_second: f64,
    end_to_end_tokens_per_second: f64,

    output: String,
}


// ============================================================
// MEMORY
// ============================================================

fn proc_kb(key: &str) -> u64 {

    fs::read_to_string(
        "/proc/self/status"
    )
    .ok()
    .and_then(|status| {

        status
            .lines()
            .find(|line| {
                line.starts_with(key)
            })
            .and_then(|line| {
                line
                    .split_whitespace()
                    .nth(1)
            })
            .and_then(|value| {
                value.parse().ok()
            })
    })
    .unwrap_or(0)
}


fn rss_mb() -> f64 {

    proc_kb("VmRSS:") as f64
    /
    1024.0
}


fn peak_rss_mb() -> f64 {

    proc_kb("VmHWM:") as f64
    /
    1024.0
}


fn file_mb(
    path: &Path,
) -> Result<f64> {

    Ok(
        fs::metadata(path)?
            .len() as f64
        /
        1024.0
        /
        1024.0
    )
}


// ============================================================
// STATISTICS
// ============================================================

fn mean(
    values: &[f64],
) -> f64 {

    if values.is_empty() {

        f64::NAN

    } else {

        values
            .iter()
            .sum::<f64>()
        /
        values.len() as f64
    }
}


fn percentile(
    values: &[f64],
    q: f64,
) -> f64 {

    if values.is_empty() {
        return f64::NAN;
    }

    let mut sorted =
        values.to_vec();

    sorted.sort_by(
        |a, b| {
            a.total_cmp(b)
        }
    );

    let index = (
        (sorted.len() - 1) as f64
        *
        q
    )
    .round() as usize;

    sorted[index]
}


// ============================================================
// SAFE HF-STYLE NO-REPEAT-NGRAM
// ============================================================
//
// For n=3:
//
// generated history:
//
//     A B C A B
//
// current last 2 tokens:
//
//     A B
//
// Previous trigram:
//
//     A B C
//
// Therefore C is banned.
//
// IMPORTANT:
// windows(ngram_size) only returns COMPLETE n-grams,
// eliminating the out-of-bounds bug from the previous version.
// ============================================================

fn banned_ngram_tokens(
    tokens: &[u32],
    ngram_size: usize,
) -> HashSet<u32> {

    let mut banned =
        HashSet::new();


    if ngram_size <= 1 {
        return banned;
    }


    let prefix_len =
        ngram_size - 1;


    // Not enough history even to form
    // the current N-1 token prefix.
    if tokens.len() < prefix_len {
        return banned;
    }


    let current_prefix_start =
        tokens.len()
        -
        prefix_len;


    let current_prefix =
        &tokens[
            current_prefix_start..
        ];


    // Only complete historical n-grams.
    for window
        in
        tokens.windows(
            ngram_size
        )
    {

        let historical_prefix =
            &window[
                ..prefix_len
            ];


        if historical_prefix
            ==
            current_prefix
        {

            let continuation =
                window[
                    prefix_len
                ];


            banned.insert(
                continuation
            );
        }
    }


    banned
}


// ============================================================
// GREEDY ARGMAX + N-GRAM BLOCKING
// ============================================================

fn greedy_next_token(
    logits: &Tensor,
    generated_tokens: &[u32],
) -> Result<u32> {

    let values =
        logits
            .to_dtype(
                DType::F32
            )?
            .to_vec1::<f32>()?;


    let banned =
        banned_ngram_tokens(
            generated_tokens,
            NO_REPEAT_NGRAM_SIZE,
        );


    let mut best_id:
        Option<u32>
        =
        None;


    let mut best_value =
        f32::NEG_INFINITY;


    for (
        token_id,
        value,
    )
    in
    values
        .iter()
        .enumerate()
    {

        let token_id =
            token_id as u32;


        if banned.contains(
            &token_id
        ) {
            continue;
        }


        if best_id.is_none()
            ||
            *value > best_value
        {

            best_id =
                Some(
                    token_id
                );

            best_value =
                *value;
        }
    }


    best_id.ok_or_else(
        || {
            E::msg(
                "No valid token after n-gram filtering"
            )
        }
    )
}


// ============================================================
// FINAL DECODING
// ============================================================

fn decode_output(
    tokenizer: &Tokenizer,
    ids: &[u32],
) -> String {

    tokenizer
        .decode(
            ids,
            true,
        )
        .unwrap_or_default()
        .replace(
            BULLET_TOKEN,
            "\n- ",
        )
        .trim()
        .to_string()
}


// ============================================================
// STREAMING
// ============================================================
//
// Decode cumulatively rather than decoding each SentencePiece
// token independently.
//
// This preserves proper whitespace.
// ============================================================

fn stream_update(
    tokenizer: &Tokenizer,
    generated: &[u32],
    previous: &mut String,
) -> Result<()> {

    let current =
        decode_output(
            tokenizer,
            generated,
        );


    if current.starts_with(
        previous.as_str()
    ) {

        let new_text =
            &current[
                previous.len()..
            ];


        if !new_text.is_empty() {

            print!(
                "{new_text}"
            );

            io::stdout()
                .flush()?;
        }

    } else {

        // Tokenizer normalization changed
        // previous whitespace.
        //
        // This affects display only.

        print!(
            "\r{current}"
        );

        io::stdout()
            .flush()?;
    }


    *previous =
        current;


    Ok(())
}


// ============================================================
// ONE INFERENCE REQUEST
// ============================================================

fn run_one(

    model:
        &mut t5::T5ForConditionalGeneration,

    config:
        &t5::Config,

    tokenizer:
        &Tokenizer,

    device:
        &Device,

    name:
        &str,

    text:
        &str,

    checkpoint_size_mb:
        f64,

    show_stream:
        bool,

) -> Result<BenchResult> {


    // ========================================================
    // CRITICAL:
    // CLEAR CACHE BETWEEN REQUESTS
    // ========================================================

    model.clear_kv_cache();


    // ========================================================
    // TOKENIZE ENCODER INPUT
    // ========================================================

    let source =
        build_text(
            text
        );


    let encoded =
        tokenizer
            .encode(
                source,
                true,
            )
            .map_err(
                E::msg
            )?;


    let source_ids =
        encoded
            .get_ids()
            .to_vec();


    let input_tokens =
        source_ids.len();


    let input_tensor =
        Tensor::new(
            source_ids.as_slice(),
            device,
        )?
        .unsqueeze(0)?;


    // ========================================================
    // START TIMER
    // ========================================================

    let total_start =
        Instant::now();


    // ========================================================
    // ENCODER PREFILL
    // ========================================================

    let prefill_start =
        Instant::now();


    let encoder_output =
        model.encode(
            &input_tensor
        )?;


    let encoder_prefill =
        prefill_start.elapsed();


    // ========================================================
    // DECODER START
    // ========================================================

    let decoder_start =
        config
            .decoder_start_token_id
            .unwrap_or(
                config.pad_token_id
            )
            as u32;


    // Complete decoder history.
    let mut decoder_history =
        vec![
            decoder_start
        ];


    // Actual generated tokens,
    // excluding decoder-start token.
    let mut generated_tokens:
        Vec<u32>
        =
        Vec::new();


    let mut token_ready_times:
        Vec<Duration>
        =
        Vec::new();


    let mut first_decoder_step =
        Duration::ZERO;


    let mut reached_eos =
        false;


    let mut previous_stream_text =
        String::new();


    if show_stream {

        println!(
            "\nSTREAM:"
        );

        io::stdout()
            .flush()?;
    }


    // ========================================================
    // DECODER LOOP
    // ========================================================

    for step
        in
        0..MAX_NEW_TOKENS
    {

        // ----------------------------------------------------
        // WITH KV CACHE:
        //
        // step 0:
        //     decoder-start token
        //
        // later:
        //     only newest decoder token
        //
        // ----------------------------------------------------

        let decoder_input =
            if step == 0
                ||
                !config.use_cache
            {

                Tensor::new(
                    decoder_history
                        .as_slice(),
                    device,
                )?
                .unsqueeze(0)?

            } else {

                let last =
                    *decoder_history
                        .last()
                        .unwrap();


                Tensor::new(
                    &[last],
                    device,
                )?
                .unsqueeze(0)?
            };


        let step_start =
            Instant::now();


        let logits =
            model
                .decode(
                    &decoder_input,
                    &encoder_output,
                )?
                .squeeze(0)?
                .to_dtype(
                    DType::F32
                )?;


        // ----------------------------------------------------
        // GREEDY DECODING
        // +
        // no_repeat_ngram_size=3
        // ----------------------------------------------------

        let next_id =
            greedy_next_token(
                &logits,
                &decoder_history,
            )?;


        let step_elapsed =
            step_start.elapsed();


        if step == 0 {

            first_decoder_step =
                step_elapsed;
        }


        token_ready_times.push(
            total_start.elapsed()
        );


        // ----------------------------------------------------
        // EOS
        // ----------------------------------------------------

        if next_id as usize
            ==
            config.eos_token_id
        {

            reached_eos =
                true;

            break;
        }


        // ----------------------------------------------------
        // UPDATE HISTORY
        // ----------------------------------------------------

        decoder_history.push(
            next_id
        );


        generated_tokens.push(
            next_id
        );


        // ----------------------------------------------------
        // STREAM
        // ----------------------------------------------------

        if show_stream {

            stream_update(

                tokenizer,

                &generated_tokens,

                &mut previous_stream_text,

            )?;
        }
    }


    if show_stream {
        println!();
    }


    let overall =
        total_start.elapsed();


    // ========================================================
    // FINAL OUTPUT
    // ========================================================

    let output =
        decode_output(
            tokenizer,
            &generated_tokens,
        );


    // ========================================================
    // TTFT
    // ========================================================

    let ttft =
        token_ready_times
            .first()
            .copied()
            .unwrap_or_default();


    // ========================================================
    // INTER-TOKEN LATENCIES
    // ========================================================

    let gaps_ms:
        Vec<f64>
        =
        token_ready_times
            .windows(2)
            .map(
                |window| {

                    (
                        window[1]
                        -
                        window[0]
                    )
                    .as_secs_f64()
                    *
                    1000.0
                }
            )
            .collect();


    let after_first =
        (
            overall.as_secs_f64()
            -
            ttft.as_secs_f64()
        )
        .max(0.0);


    let tokens_after_first =
        token_ready_times
            .len()
            .saturating_sub(1);


    let decode_tokens_per_second =
        if after_first > 0.0 {

            tokens_after_first as f64
            /
            after_first

        } else {

            f64::NAN
        };


    let end_to_end_tokens_per_second =
        if overall.as_secs_f64() > 0.0 {

            token_ready_times.len() as f64
            /
            overall.as_secs_f64()

        } else {

            f64::NAN
        };


    let max_itl_ms =
        if gaps_ms.is_empty() {

            f64::NAN

        } else {

            gaps_ms
                .iter()
                .copied()
                .fold(
                    f64::NEG_INFINITY,
                    f64::max,
                )
        };


    Ok(
        BenchResult {

            test:
                name.to_string(),

            input_words:
                text
                    .split_whitespace()
                    .count(),

            input_tokens,

            output_tokens:
                generated_tokens.len(),

            reached_eos,

            checkpoint_size_mb,

            rss_mb:
                rss_mb(),

            peak_rss_mb:
                peak_rss_mb(),

            encoder_prefill_seconds:
                encoder_prefill
                    .as_secs_f64(),

            first_decoder_step_seconds:
                first_decoder_step
                    .as_secs_f64(),

            ttft_seconds:
                ttft
                    .as_secs_f64(),

            mean_itl_ms:
                mean(
                    &gaps_ms
                ),

            median_itl_ms:
                percentile(
                    &gaps_ms,
                    0.50,
                ),

            p95_itl_ms:
                percentile(
                    &gaps_ms,
                    0.95,
                ),

            max_itl_ms,

            overall_latency_seconds:
                overall
                    .as_secs_f64(),

            decode_tokens_per_second,

            end_to_end_tokens_per_second,

            output,
        }
    )
}


// ============================================================
// MAIN
// ============================================================

fn main() -> Result<()> {

    println!(
        "============================================================"
    );

    println!(
        "Rust + Candle T5 FP32 parity benchmark"
    );

    println!(
        "Model: {MODEL_ID}"
    );

    println!(
        "KV cache reset: ENABLED PER REQUEST"
    );

    println!(
        "no_repeat_ngram_size: {}",
        NO_REPEAT_NGRAM_SIZE
    );

    println!(
        "============================================================"
    );


    let device =
        Device::Cpu;


    // ========================================================
    // HUGGING FACE MODEL FILES
    // ========================================================

    let api =
        Api::new()?;


    let repo =
        api.repo(

            Repo::new(
                MODEL_ID.into(),
                RepoType::Model,
            )
        );


    let config_path =
        repo
            .get(
                "config.json"
            )
            .context(
                "config.json"
            )?;


    let tokenizer_path =
        repo
            .get(
                "tokenizer.json"
            )
            .context(
                "tokenizer.json"
            )?;


    let weights_path =
        repo
            .get(
                "model.safetensors"
            )
            .context(
                "model.safetensors"
            )?;


    let checkpoint_size_mb =
        file_mb(
            &weights_path
        )?;


    // ========================================================
    // CONFIG
    // ========================================================

    let mut config:
        t5::Config
        =
        serde_json::from_str(

            &fs::read_to_string(
                &config_path
            )?

        )?;


    config.use_cache =
        true;


    println!(
        "decoder_start_token_id: {:?}",
        config.decoder_start_token_id
    );

    println!(
        "pad_token_id: {}",
        config.pad_token_id
    );

    println!(
        "eos_token_id: {}",
        config.eos_token_id
    );

    println!(
        "use_cache: {}",
        config.use_cache
    );


    // ========================================================
    // TOKENIZER
    // ========================================================

    let mut tokenizer =
        Tokenizer::from_file(
            &tokenizer_path
        )
        .map_err(
            E::msg
        )?;


    tokenizer.with_padding(
        None
    );


    tokenizer
        .with_truncation(
            None
        )
        .map_err(
            E::msg
        )?;


    println!(
        "<BULLET> id: {:?}",
        tokenizer
            .token_to_id(
                BULLET_TOKEN
            )
    );


    println!(
        "Checkpoint: {:.2} MB",
        checkpoint_size_mb
    );


    println!(
        "RSS before load: {:.2} MB",
        rss_mb()
    );


    // ========================================================
    // LOAD FP32 SAFETENSORS
    // ========================================================

    let load_start =
        Instant::now();


    let vb =
        unsafe {

            VarBuilder::
                from_mmaped_safetensors(

                    &[weights_path],

                    DType::F32,

                    &device,

                )?
        };


    let mut model =
        t5::
            T5ForConditionalGeneration::
            load(
                vb,
                &config,
            )?;


    println!(
        "Model load: {:.3} s",
        load_start
            .elapsed()
            .as_secs_f64()
    );


    println!(
        "RSS after load: {:.2} MB",
        rss_mb()
    );


    // ========================================================
    // WARMUP
    // ========================================================

    println!(
        "\nWarmup..."
    );


    let warm =
        test_texts()[0];


    let _ =
        run_one(

            &mut model,

            &config,

            &tokenizer,

            &device,

            warm.0,

            warm.1,

            checkpoint_size_mb,

            false,

        )?;


    // Explicitly clean after warmup.
    model.clear_kv_cache();


    println!(
        "Warmup complete. KV cache cleared."
    );


    // ========================================================
    // RUN FIVE TESTS
    // ========================================================

    let mut results =
        Vec::new();


    for (
        name,
        text,
    )
    in test_texts()
    {

        println!(
            "\n{}",
            "=".repeat(100)
        );


        println!(
            "TEST: {name}"
        );


        println!(
            "{}",
            "=".repeat(100)
        );


        let result =
            run_one(

                &mut model,

                &config,

                &tokenizer,

                &device,

                name,

                text,

                checkpoint_size_mb,

                true,

            )?;


        println!(
            "\nMETRICS"
        );


        println!(
            "input tokens:       {}",
            result.input_tokens
        );


        println!(
            "output tokens:      {}",
            result.output_tokens
        );


        println!(
            "reached EOS:        {}",
            result.reached_eos
        );


        println!(
            "encoder prefill:    {:.4} s",
            result.encoder_prefill_seconds
        );


        println!(
            "first decoder step: {:.4} s",
            result.first_decoder_step_seconds
        );


        println!(
            "TTFT:               {:.4} s",
            result.ttft_seconds
        );


        println!(
            "mean ITL:           {:.2} ms",
            result.mean_itl_ms
        );


        println!(
            "median ITL:         {:.2} ms",
            result.median_itl_ms
        );


        println!(
            "p95 ITL:            {:.2} ms",
            result.p95_itl_ms
        );


        println!(
            "total latency:      {:.4} s",
            result.overall_latency_seconds
        );


        println!(
            "decode tok/s:       {:.2}",
            result.decode_tokens_per_second
        );


        results.push(
            result
        );
    }


    // ========================================================
    // REPORTS
    // ========================================================

    fs::create_dir_all(
        "reports"
    )?;


    fs::write(

        "reports/candle_fp32_results.json",

        serde_json::
            to_string_pretty(
                &results
            )?,

    )?;


    let mut csv =
        String::from(

            "test,input_words,input_tokens,output_tokens,reached_eos,checkpoint_size_mb,rss_mb,peak_rss_mb,encoder_prefill_seconds,first_decoder_step_seconds,ttft_seconds,mean_itl_ms,median_itl_ms,p95_itl_ms,max_itl_ms,overall_latency_seconds,decode_tokens_per_second,end_to_end_tokens_per_second\n"

        );


    for r
        in
        &results
    {

        csv.push_str(

            &format!(

                "{},{},{},{},{},{:.4},{:.4},{:.4},{:.6},{:.6},{:.6},{:.4},{:.4},{:.4},{:.4},{:.6},{:.4},{:.4}\n",

                r.test,

                r.input_words,

                r.input_tokens,

                r.output_tokens,

                r.reached_eos,

                r.checkpoint_size_mb,

                r.rss_mb,

                r.peak_rss_mb,

                r.encoder_prefill_seconds,

                r.first_decoder_step_seconds,

                r.ttft_seconds,

                r.mean_itl_ms,

                r.median_itl_ms,

                r.p95_itl_ms,

                r.max_itl_ms,

                r.overall_latency_seconds,

                r.decode_tokens_per_second,

                r.end_to_end_tokens_per_second,

            )
        );
    }


    fs::write(

        "reports/candle_fp32_results.csv",

        csv,

    )?;


    println!(
        "\nReports written."
    );


    Ok(())
}
'''


with open(
    "/content/candle_t5_native/src/main.rs",
    "w",
) as f:

    f.write(
        rust_source
    )


print(
    "Updated Cell 3 written successfully."
)

print(
    "Fixes:"
)

print(
    "  ✓ KV cache reset per request"
)

print(
    "  ✓ safe no_repeat_ngram_size=3"
)

print(
    "  ✓ no out-of-bounds indexing"
)

print(
    "  ✓ greedy decoding"
)

print(
    "  ✓ cached decoder"
)

print(
    "  ✓ cumulative streaming"
)

Updated Cell 3 written successfully.
Fixes:
  ✓ KV cache reset per request
  ✓ safe no_repeat_ngram_size=3
  ✓ no out-of-bounds indexing
  ✓ greedy decoding
  ✓ cached decoder
  ✓ cumulative streaming


In [ ]:
# ============================================================
# CELL 5 — RECOMPILE
# ============================================================

%cd /content/candle_t5_native

!cargo build --release

/content/candle_t5_native
   Compiling candle-t5-native v0.1.0 (/content/candle_t5_native)
    Finished `release` profile [optimized] target(s) in 18.21s


In [ ]:
# CELL 5 — verify native executable
!ls -lh target/release/candle-t5-native
!file target/release/candle-t5-native
!ldd target/release/candle-t5-native | head -30

-rwxr-xr-x 2 root root 13M Sep 14 08:33 target/release/candle-t5-native
target/release/candle-t5-native: ELF 64-bit LSB pie executable, x86-64, version 1 (SYSV), dynamically linked, interpreter /lib64/ld-linux-x86-64.so.2, for GNU/Linux 3.2.0, BuildID[sha1]=f30aedc10b2f9c82d471935d45511c30485c10a9, not stripped
	linux-vdso.so.1 (0x00007ffc4193b000)
	libssl.so.3 => /lib/x86_64-linux-gnu/libssl.so.3 (0x0000781131f02000)
	libcrypto.so.3 => /lib/x86_64-linux-gnu/libcrypto.so.3 (0x00007811319ee000)
	libstdc++.so.6 => /lib/x86_64-linux-gnu/libstdc++.so.6 (0x0000781131770000)
	libgcc_s.so.1 => /lib/x86_64-linux-gnu/libgcc_s.so.1 (0x0000781131742000)
	libm.so.6 => /lib/x86_64-linux-gnu/libm.so.6 (0x0000781131659000)
	libc.so.6 => /lib/x86_64-linux-gnu/libc.so.6 (0x0000781131445000)
	/lib64/ld-linux-x86-64.so.2 (0x0000781132999000)


In [ ]:
# ============================================================
# CELL 6 — RUN CORRECTED NATIVE BENCHMARK
# ============================================================

%cd /content/candle_t5_native

!mkdir -p reports

!./target/release/candle-t5-native \
    2>&1 | tee reports/full_run.log

/content/candle_t5_native
Rust + Candle T5 FP32 parity benchmark
Model: JayShah07/falconai-text-bullet-t5
KV cache reset: ENABLED PER REQUEST
no_repeat_ngram_size: 3
decoder_start_token_id: Some(0)
pad_token_id: 0
eos_token_id: 1
use_cache: true
<BULLET> id: Some(32100)
Checkpoint: 230.78 MB
RSS before load: 44.05 MB
Model load: 0.287 s
RSS after load: 275.44 MB

Warmup...
Warmup complete. KV cache cleared.

TEST: 01_very_short

STREAM:
-  Apex Systems reported quarterly revenue of $1.8 billion, up 9% year over year.
-  Operating profit increased 6% to $240 million, while management maintained its full-year guidance.

METRICS
input tokens:       242
output tokens:      39
reached EOS:        true
encoder prefill:    0.2809 s
first decoder step: 0.0540 s
TTFT:               0.3350 s
mean ITL:           62.12 ms
median ITL:         55.52 ms
p95 ITL:            88.69 ms
total latency:      2.7576 s
decode tok/s:       16.10

TEST: 02_short

STREAM:
-  Vertex Systems reported quarterly rev

In [ ]:
# ============================================================
# CELL 7 — LOAD RESULTS
# ============================================================

import pandas as pd

results = pd.read_csv(
    "/content/candle_t5_native/reports/candle_fp32_results.csv"
)

display(results)

,test,input_words,input_tokens,output_tokens,reached_eos,checkpoint_size_mb,rss_mb,peak_rss_mb,encoder_prefill_seconds,first_decoder_step_seconds,ttft_seconds,mean_itl_ms,median_itl_ms,p95_itl_ms,max_itl_ms,overall_latency_seconds,decode_tokens_per_second,end_to_end_tokens_per_second
0,01_very_short,26,242,39,True,230.7764,280.6016,506.1484,0.280887,0.054048,0.334972,62.1184,55.5193,88.6862,95.4853,2.757616,16.0981,14.5053
1,02_short,66,291,90,True,230.7764,282.9609,506.1484,0.583226,0.091085,0.674338,71.7700,63.0603,97.2035,102.5035,7.133691,13.9333,12.7564
2,03_medium,102,348,149,True,230.7764,285.4570,506.1484,0.425876,0.072993,0.498925,81.2590,71.4017,114.3309,123.5143,12.606552,12.3063,11.8986
3,04_long,150,402,187,True,230.7764,287.7695,506.1484,0.503828,0.074634,0.578514,86.0613,77.7711,125.8434,137.8425,16.672036,11.6196,11.2764
4,05_very_long,288,610,256,False,230.7764,293.9258,506.1484,0.905721,0.100878,1.006640,128.7732,106.1351,182.6840,312.9597,33.844181,7.7655,7.5641


In [ ]:
# ============================================================
# CELL 8 — SERVING TABLE
# ============================================================

cols = [
    "test",
    "input_words",
    "input_tokens",
    "output_tokens",
    "reached_eos",
    "checkpoint_size_mb",
    "rss_mb",
    "peak_rss_mb",
    "encoder_prefill_seconds",
    "first_decoder_step_seconds",
    "ttft_seconds",
    "mean_itl_ms",
    "median_itl_ms",
    "p95_itl_ms",
    "max_itl_ms",
    "overall_latency_seconds",
    "decode_tokens_per_second",
    "end_to_end_tokens_per_second",
]

display(
    results[cols]
)

,test,input_words,input_tokens,output_tokens,reached_eos,checkpoint_size_mb,rss_mb,peak_rss_mb,encoder_prefill_seconds,first_decoder_step_seconds,ttft_seconds,mean_itl_ms,median_itl_ms,p95_itl_ms,max_itl_ms,overall_latency_seconds,decode_tokens_per_second,end_to_end_tokens_per_second
0,01_very_short,26,242,39,True,230.7764,280.6016,506.1484,0.280887,0.054048,0.334972,62.1184,55.5193,88.6862,95.4853,2.757616,16.0981,14.5053
1,02_short,66,291,90,True,230.7764,282.9609,506.1484,0.583226,0.091085,0.674338,71.7700,63.0603,97.2035,102.5035,7.133691,13.9333,12.7564
2,03_medium,102,348,149,True,230.7764,285.4570,506.1484,0.425876,0.072993,0.498925,81.2590,71.4017,114.3309,123.5143,12.606552,12.3063,11.8986
3,04_long,150,402,187,True,230.7764,287.7695,506.1484,0.503828,0.074634,0.578514,86.0613,77.7711,125.8434,137.8425,16.672036,11.6196,11.2764
4,05_very_long,288,610,256,False,230.7764,293.9258,506.1484,0.905721,0.100878,1.006640,128.7732,106.1351,182.6840,312.9597,33.844181,7.7655,7.5641


In [ ]:
# CELL 9 — summary
summary=pd.DataFrame([{'runtime':'Rust_Candle_FP32','checkpoint_size_mb':results.checkpoint_size_mb.iloc[0],'max_peak_rss_mb':results.peak_rss_mb.max(),'avg_prefill_seconds':results.encoder_prefill_seconds.mean(),'avg_ttft_seconds':results.ttft_seconds.mean(),'median_ttft_seconds':results.ttft_seconds.median(),'avg_mean_itl_ms':results.mean_itl_ms.mean(),'avg_p95_itl_ms':results.p95_itl_ms.mean(),'avg_total_latency_seconds':results.overall_latency_seconds.mean(),'avg_decode_tokens_per_second':results.decode_tokens_per_second.mean(),'avg_end_to_end_tokens_per_second':results.end_to_end_tokens_per_second.mean()}])
display(summary)
summary.to_csv('/content/candle_t5_native/reports/candle_fp32_summary.csv',index=False)

,runtime,checkpoint_size_mb,max_peak_rss_mb,avg_prefill_seconds,avg_ttft_seconds,median_ttft_seconds,avg_mean_itl_ms,avg_p95_itl_ms,avg_total_latency_seconds,avg_decode_tokens_per_second,avg_end_to_end_tokens_per_second
0,Rust_Candle_FP32,230.7764,506.1484,0.539908,0.618678,0.578514,85.99638,121.7496,14.602815,12.34456,11.60016


In [ ]:
# CELL 10 — TTFT scaling
display(results[['test','input_tokens','encoder_prefill_seconds','first_decoder_step_seconds','ttft_seconds']])

,test,input_tokens,encoder_prefill_seconds,first_decoder_step_seconds,ttft_seconds
0,01_very_short,242,0.280887,0.054048,0.334972
1,02_short,291,0.583226,0.091085,0.674338
2,03_medium,348,0.425876,0.072993,0.498925
3,04_long,402,0.503828,0.074634,0.578514
4,05_very_long,610,0.905721,0.100878,1.006640


In [ ]:
# CELL 11 — streaming smoothness
display(results[['test','output_tokens','mean_itl_ms','median_itl_ms','p95_itl_ms','max_itl_ms','decode_tokens_per_second']])

,test,output_tokens,mean_itl_ms,median_itl_ms,p95_itl_ms,max_itl_ms,decode_tokens_per_second
0,01_very_short,39,62.1184,55.5193,88.6862,95.4853,16.0981
1,02_short,90,71.7700,63.0603,97.2035,102.5035,13.9333
2,03_medium,149,81.2590,71.4017,114.3309,123.5143,12.3063
3,04_long,187,86.0613,77.7711,125.8434,137.8425,11.6196
4,05_very_long,256,128.7732,106.1351,182.6840,312.9597,7.7655


In [ ]:
# CELL 12 — show full generated outputs
import json
rows=json.load(open('/content/candle_t5_native/reports/candle_fp32_results.json'))
for r in rows:
 print('\n'+'='*80+'\n'+r['test']+'\n'+'='*80);print(r['output'])


01_very_short
-  Apex Systems reported quarterly revenue of $1.8 billion, up 9% year over year.
-  Operating profit increased 6% to $240 million, while management maintained its full-year guidance.

02_short
-  Vertex Systems reported quarterly revenue of $3.8 billion, up 16% from the same period last year.
-  Cloud-service revenue increased 28%, while legacy hardware sales declined 7%.
-  Operating profit rose from $410 million to $475 million, although operating margin decreased from 18.1% to 17.4% because of higher infrastructure and energy costs.
-  The company added 820,000 paying customers and said demand remained strong in North America and Asia.

03_medium
-  Northstar Technologies reported second-quarter revenue of $8.7 billion, an increase of 18% year over year.
-  Growth was driven primarily by cloud infrastructure and enterprise subscriptions.
-  Cloud revenue rose 31%, while the older hardware business declined 9%.
-  Operating profit increased from $940 million to $1.12 

In [ ]:
# CELL 13 — independent process memory/wall-time
for line in open('/content/candle_t5_native/reports/full_run.log',errors='ignore'):
 if any(k in line for k in ['Maximum resident set size','Elapsed (wall clock)','User time','System time']): print(line.strip())

In [ ]:
# CELL 14 — output files
from pathlib import Path
!du -h /content/candle_t5_native/target/release/candle-t5-native
for p in sorted(Path('/content/candle_t5_native/reports').glob('*')):print(p)

13M	/content/candle_t5_native/target/release/candle-t5-native
/content/candle_t5_native/reports/candle_fp32_results.csv
/content/candle_t5_native/reports/candle_fp32_results.json
/content/candle_t5_native/reports/candle_fp32_summary.csv
/content/candle_t5_native/reports/full_run.log


## Next step

If all five outputs are sensible, Candle has successfully executed the fine-tuned T5 checkpoint directly from Hugging Face safetensors. The next experiment is Candle-native INT8 and a direct TTFT/ITL/RSS comparison with `TorchAO_INT8_WEIGHT_ONLY`.